**Theme:**

*“The best model is rarely the last one.”*

# Imports

In [1]:
import torch 
import torch.nn as nn 

torch.__version__

'2.8.0+cu129'

In [4]:
from torchvision import datasets, transforms 
from torch.utils.data import DataLoader, random_split

# Model checkpointin, Early stopping and Training like production

In [5]:
# Preparing train and validation data
transform = transforms.ToTensor() 

train_data_full = datasets.MNIST(root='data', train=True, download=True, transform=transform)  

train_size = int(0.8 * len(train_data_full))   
val_size = len(train_data_full) - train_size 

train_data, val_data = random_split(train_data_full, [train_size, val_size])  

train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
val_loader = DataLoader(val_data, batch_size=64, shuffle=False) 


# Neural Network
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128,10),
        )
    
    def forward(self, x):
        return self.net(x) 

## Save the BEST Model (Checkpointing)

In [7]:
model = MLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  

In [9]:
best_val_loss = float('inf')

for epoch in range(30):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        outputs = model(images) 
        loss = loss_fn(outputs, labels) 

        optimizer.zero_grad()  
        loss.backward()
        optimizer.step()

        train_loss += loss.item()  

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images) 
            loss = loss_fn(outputs,labels) 

            val_loss += loss.item() 

    train_loss /= len(train_loader)  
    val_loss /= len(val_loader) 

    # showing loss in each 5 epoch
    if (epoch+1) % 5 == 0:
        print(f"Epoch: {epoch+1} | TrainLoss: {train_loss:.4f} | ValLoss: {val_loss:.4f}")

    # checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),'best_mnist_model.pth')

Epoch: 5 | TrainLoss: 0.0644 | ValLoss: 0.0965
Epoch: 10 | TrainLoss: 0.0253 | ValLoss: 0.0909
Epoch: 15 | TrainLoss: 0.0099 | ValLoss: 0.0947
Epoch: 20 | TrainLoss: 0.0062 | ValLoss: 0.1022
Epoch: 25 | TrainLoss: 0.0010 | ValLoss: 0.1121
Epoch: 30 | TrainLoss: 0.0007 | ValLoss: 0.1154


📌 Answer:

- Why save based on **validation**, not **training loss**?

## Early Stopping (Stop wasting time)

In [11]:
model = MLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  

In [12]:
best_val_loss = float('inf')
patience = 3 
counter = 0

for epoch in range(30):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        outputs = model(images) 
        loss = loss_fn(outputs, labels) 

        optimizer.zero_grad()  
        loss.backward()
        optimizer.step()

        train_loss += loss.item()  

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images) 
            loss = loss_fn(outputs,labels) 

            val_loss += loss.item() 

    train_loss /= len(train_loader)  
    val_loss /= len(val_loader) 

    # showing loss in each 5 epoch
    if (epoch+1) % 5 == 0:
        print(f"Epoch: {epoch+1} | TrainLoss: {train_loss:.4f} | ValLoss: {val_loss:.4f}")

    # checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(),'best_mnist_model.pth')
        counter = 0 
    else: 
        counter += 1 
    
    if counter >= patience:
        print("Early stopping triggered at epoch:",epoch+1)  
        break

Epoch: 5 | TrainLoss: 0.0767 | ValLoss: 0.0940
Epoch: 10 | TrainLoss: 0.0281 | ValLoss: 0.0807
Early stopping triggered at epoch: 14


📌 Answer:

- What problem does early stopping solve?

## Load the best model and Evaluate

In [14]:
# Test data preparation 
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform) 

test_loader = DataLoader(test_data, batch_size=64, shuffle=False) 

In [15]:
images,labels = next(iter(test_loader))
labels.shape

torch.Size([64])

In [20]:
total = 0 
correct = 0 

best_model = MLP()
best_model.load_state_dict(torch.load('best_mnist_model.pth')) 
best_model.eval()

with torch.no_grad():
    for images,labels in test_loader:
        outputs = best_model(images) 
        loss = loss_fn(outputs, labels) 

        preds = outputs.argmax(dim=1) 
        
        correct += (preds==labels).sum().item() 
        total += len(labels)
    
    print(f"Best Model Test Accuracy: {correct*100/total:.3f} %")

Best Model Test Accuracy: 97.620 %


📌 Answer:

- Why might this accuracy be better than the last epoch’s model?

---

**TASK** — Professional Mental Model (VERY IMPORTANT)

Answer in words:

- Why is validation loss the decision-maker?

- Why is training “too long” a bad idea?

- What would happen if we never used early stopping?

# Interaction

- Write down one confusion.
- Then read the notebook carefully (Re-read)
- Write down the confusion's answer on own and then compare with 'LLM's/professionals answer.